In [1]:
from fastembed import LateInteractionTextEmbedding
from qdrant_client import QdrantClient, models
import time
import plotly.io as pio
import numpy as np
from fastembed import LateInteractionTextEmbedding
from fastembed import TextEmbedding
from qdrant_client import QdrantClient, models
import time

pio.renderers.default = "notebook"

2026-02-24 11:30:54.632615640 [W:onnxruntime:Default, device_discovery.cc:211 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:91 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


### Embedding Models

In [2]:
colbert_model = LateInteractionTextEmbedding(
    model_name="colbert-ir/colbertv2.0"
)

In [3]:
dense_model = TextEmbedding("BAAI/bge-small-en-v1.5")

### Qdrant

In [4]:
collection_name = "colbert-tests"
dense_vector_name = "BAAI-bge-small-en-v1.5"
colbert_vector_name = "colbert-ir-colbertv2.0"

In [5]:
client = QdrantClient("http://localhost:6333")
client.delete_collection(collection_name)

False

In [6]:
client.create_collection(
    collection_name,
    vectors_config={
        dense_vector_name: models.VectorParams(
            size=dense_model.embedding_size,
            distance=models.Distance.COSINE,
        ),
        colbert_vector_name: models.VectorParams(
            size=colbert_model.embedding_size,
            distance=models.Distance.DOT,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM,
            ),
            hnsw_config=models.HnswConfigDiff(m=0),
        ),
    },
)

True

In [7]:
documents = [
    "Qdrant is a vector database designed for similarity search applications",
    "SQL databases use structured tables with predefined schemas for data storage",
    "Using Qdrant you can store embeddings and perform efficient searches",
    "Traditional SQL queries filter data using exact matches and joins",
    "Qdrant supports multi-vector configurations for late interaction models like ColBERT",
    "SQL performs well for transactional workloads but lacks semantic search capabilities",
    "The Qdrant client allows you to create collections with custom distance metrics",
    "Migrating from SQL to vector databases enables similarity-based retrieval at scale",
    "Qdrant's MaxSim comparator enables token-level similarity scoring for multi-vectors",
    "SQL databases struggle with high-dimensional embeddings unlike specialized vector stores",
]

In [8]:
client.upsert(
    collection_name,
    points=[
        models.PointStruct(
            id=i,
            vector={
                dense_vector_name: next(
                    dense_model.passage_embed([document])
                ),
                colbert_vector_name: next(
                    colbert_model.passage_embed([document])
                ),
            },
            payload={"text": document},
        )
        for i, document in enumerate(documents, start=1)
    ],
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

### Query

In [9]:
def colbert_query(q: str, limit: int = 5) -> list[dict]:
    start_time = time.monotonic()
    embedding = next(colbert_model.query_embed(q))
    end_time = time.monotonic()
    print("ColBERT vector generation time:", end_time - start_time)

    start_time = time.monotonic()
    result = client.query_points(
        collection_name,
        query=embedding,
        using=colbert_vector_name,
        limit=limit,
        with_payload=True,
    )
    end_time = time.monotonic()
    print("Query time:", end_time - start_time)

    return [point.payload for point in result.points]

In [10]:
def dense_query(q: str, limit: int = 5) -> list[dict]:
    start_time = time.monotonic()
    embedding = next(dense_model.query_embed(q))
    end_time = time.monotonic()
    print("Dense vector generation time:", end_time - start_time)

    start_time = time.monotonic()
    result = client.query_points(
        collection_name,
        query=embedding,
        using=dense_vector_name,
        limit=limit,
        with_payload=True,
    )
    end_time = time.monotonic()
    print("Query time:", end_time - start_time)

    return [point.payload for point in result.points]

In [11]:
query = "search performance in Qdrant"

In [12]:
colbert_hits = colbert_query(query)
dense_hits = dense_query(query)

ColBERT vector generation time: 0.054083431999970344
Query time: 0.024384004000239656
Dense vector generation time: 0.0060486020001917495
Query time: 0.007054566000078921


In [13]:
colbert_hits

[{'text': 'Using Qdrant you can store embeddings and perform efficient searches'},
 {'text': 'Qdrant is a vector database designed for similarity search applications'},
 {'text': "Qdrant's MaxSim comparator enables token-level similarity scoring for multi-vectors"},
 {'text': 'Qdrant supports multi-vector configurations for late interaction models like ColBERT'},
 {'text': 'The Qdrant client allows you to create collections with custom distance metrics'}]

In [14]:
dense_hits

[{'text': 'Using Qdrant you can store embeddings and perform efficient searches'},
 {'text': 'Qdrant is a vector database designed for similarity search applications'},
 {'text': 'Qdrant supports multi-vector configurations for late interaction models like ColBERT'},
 {'text': 'SQL performs well for transactional workloads but lacks semantic search capabilities'},
 {'text': 'Migrating from SQL to vector databases enables similarity-based retrieval at scale'}]